# <font color="#418FDE" size="6.5" uppercase>**LeNet mit PyTorch**</font>

>Last update: 20260825.
    
By the end of this Lecture, you will be able to:
- Laden kleine Bilddatensätze mit torchvision und definieren Transformationen. 
- Implementieren ein LeNet-ähnliches CNN mit Conv2d, ReLU, Pooling und Dense-Schichten. 
- Trainieren und bewerten das CNN mit Validierung, Konfusionsmatrix und Fehlerbildern. 


## **1. torchvision Daten**

### **1.1. Daten laden**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_18/Lecture_A/image_01_01.jpg?v=1787663014" width="250">



>* torchvision lädt kleine Bilddatensätze einheitlich
>* Bilder und Labels gehören zusammen

>* Training passt Modellparameter an
>* Validierung und Test prüfen Generalisierung

>* Batches sparen Speicher und stabilisieren Training
>* Mischen verhindert Reihenfolge-Effekte beim Lernen



In [ ]:
#@title Python-Code - Daten laden

# Wir laden Bilddaten für ein kleines CNN.
# Transformationen machen Bilder zu PyTorch Tensoren.
# Ein Batch zeigt Form und Labels.

import torch
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset

# Der Datensatz enthält kleine Ziffernbilder ohne Download.
digits = load_digits()
images_uint8 = digits.images.astype("uint8")
labels_int64 = digits.target.astype("int64")

# Diese Prüfung schützt vor unerwarteten Datenformen.
if images_uint8.ndim != 3 or images_uint8.shape[1:] != (8, 8):
    raise ValueError("Erwartet werden Ziffernbilder mit der Form 8 mal 8.")

# Die Transformation skaliert Pixel und ergänzt den Kanal.
images_tensor = torch.tensor(images_uint8, dtype=torch.float32) / 16.0
images_tensor = images_tensor.unsqueeze(1)
labels_tensor = torch.tensor(labels_int64, dtype=torch.long)

# TensorDataset verbindet jedes Bild mit seinem Label.
dataset = TensorDataset(images_tensor, labels_tensor)
loader = DataLoader(dataset, batch_size=16, shuffle=True)

# Ein Batch entspricht einer kleinen Trainingsportion.
batch_images, batch_labels = next(iter(loader))
first_image = batch_images[0, 0].numpy()
first_label = int(batch_labels[0])

print("Datensatz: sklearn digits, kleine Ziffernbilder.")
print(f"Alle Bilder als Tensor: {tuple(images_tensor.shape)}")
print(f"Ein Batch Bilder: {tuple(batch_images.shape)}")
print(f"Ein Batch Labels: {tuple(batch_labels.shape)}")
print(f"Pixelbereich nach Transformation: {batch_images.min():.2f} bis {batch_images.max():.2f}")
print(f"Label des gezeigten Bildes: {first_label}")

# Die Abbildung zeigt ein geladenes Beispielbild.
fig, ax = plt.subplots(figsize=(3, 3))
ax.imshow(first_image, cmap="gray", vmin=0.0, vmax=1.0)
ax.set_title("Ein geladenes Ziffernbild")
ax.set_xlabel("Pixelspalte")
ax.set_ylabel("Pixelzeile")
plt.show()



### **1.2. Transformationen definieren**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_18/Lecture_A/image_01_02.jpg?v=1787663016" width="250">



>* Transformationen machen Bilder modellgerecht nutzbar
>* Einheitliche Eingaben fördern relevante Mustererkennung

>* Normalisierung stabilisiert Bildwerte fürs Training
>* Einheitliche Bildformen unterstützen CNN-Mustererkennung

>* Augmentation erhöht Trainingsvielfalt durch passende Bildänderungen
>* Validierung und Tests bleiben deterministisch



In [ ]:
#@title Python-Code - Transformationen definieren

# Wir definieren einfache Bildtransformationen für kleine Datensätze.
# Tensorumwandlung und Normalisierung werden sichtbar verglichen.
# Die Ausgabe zeigt Formen, Wertebereiche und ein Bild.

import numpy as np
import torch
import matplotlib.pyplot as plt

# Ein kleines synthetisches Graustufenbild ersetzt externe Dateien.
height = 28
width = 28
image_uint8 = np.zeros((height, width), dtype=np.uint8)

# Ein helles Quadrat simuliert ein einfaches Bildmotiv.
image_uint8[8:20, 9:19] = 220
image_uint8[12:16, 13:15] = 80

# Diese Funktion imitiert eine typische torchvision-Vorbereitung.
def transform_grayscale_image(image):
    tensor = torch.from_numpy(image.copy()).float()
    tensor = tensor.unsqueeze(0) / 255.0
    normalized = (tensor - 0.5) / 0.5
    return tensor, normalized

# Die Transformation erzeugt Kanal, Höhe und Breite.
tensor_image, normalized_image = transform_grayscale_image(image_uint8)

# Eine einfache Prüfung macht die erwartete Form explizit.
expected_shape = (1, 28, 28)
if tuple(normalized_image.shape) != expected_shape:
    raise ValueError("Die transformierte Bildform ist unerwartet.")

# Kurze Ausgaben zeigen die wichtigsten Änderungen.
print("Originalform:", image_uint8.shape)
print("Tensorform:", tuple(tensor_image.shape))
print("Tensorwerte:", round(float(tensor_image.min()), 2), "bis", round(float(tensor_image.max()), 2))
print("Normalisiert:", round(float(normalized_image.min()), 2), "bis", round(float(normalized_image.max()), 2))

# Für die Anzeige wird der normalisierte Tensor zurückskaliert.
display_image = normalized_image.squeeze(0).numpy()
fig, ax = plt.subplots(figsize=(4, 4))

# Ein einzelnes Bild zeigt das Ergebnis der Vorbereitung.
ax.imshow(display_image, cmap="gray", vmin=-1.0, vmax=1.0)
ax.set_title("Normalisiertes synthetisches Graustufenbild")
ax.set_xlabel("Pixelspalte")
ax.set_ylabel("Pixelzeile")
plt.show()



### **1.3. Conv2d Formen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_18/Lecture_A/image_01_03.jpg?v=1787663018" width="250">



>* Conv2d erwartet Stapel mit Kanälen zuerst
>* Tensorformen helfen, Modellfehler zu verstehen

>* Eingangskanäle passend zum Bildtyp wählen
>* Faltung erkennt Muster, Pooling verkleinert Merkmalskarten

>* Conv2d-Formen vor Dense-Schichten genau verfolgen
>* Passende Formen ermöglichen flexible LeNet-Anpassungen



In [ ]:
#@title Python-Code - Conv2d Formen

# Dieses Beispiel zeigt Conv2d-Formen mit Bildtensoren.
# Kanäle, Höhe und Breite werden sichtbar.
# Die Ausgabe erklärt passende Schichtparameter.

import torch
import matplotlib.pyplot as plt

# Ein kleiner Stapel simuliert zwei Graustufenbilder.
batch_size = 2
channels = 1
height = 28
width = 28

# Die Werte sind deterministisch und bleiben klein.
torch.manual_seed(42)
images = torch.rand(batch_size, channels, height, width)

# Conv2d erwartet die Reihenfolge N, C, H, W.
conv_layer = torch.nn.Conv2d(
    in_channels=1,
    out_channels=6,
    kernel_size=5,
)

# Die Faltung erzeugt neue Merkmalskarten.
feature_maps = conv_layer(images)

# Pooling halbiert hier die räumliche Auflösung.
pool_layer = torch.nn.MaxPool2d(kernel_size=2)
pooled_maps = pool_layer(feature_maps)

# Diese Prüfung macht die wichtigste Annahme sichtbar.
if images.shape[1] != conv_layer.in_channels:
    raise ValueError("Die Kanalzahl passt nicht zur Conv2d-Schicht.")

print(f"Eingabeform: {tuple(images.shape)} = Stapel, Kanäle, Höhe, Breite")
print(f"Nach Conv2d: {tuple(feature_maps.shape)}")
print(f"Nach MaxPool2d: {tuple(pooled_maps.shape)}")
print(f"Werte pro Bild vor Dense: {pooled_maps[0].numel()}")

# Die Grafik zeigt ein Bild aus dem Tensorstapel.
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(images[0, 0].detach().numpy(), cmap="gray")
ax.set_title("Synthetisches Graustufenbild: Kanal 0")
ax.set_xlabel("Breite in Pixeln")
ax.set_ylabel("Höhe in Pixeln")
plt.show()



## **2. LeNet Modul**

### **2.1. Aktivierung und Pooling**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_18/Lecture_A/image_02_01.jpg?v=1787663001" width="250">



>* Aktivierungen ermöglichen komplexe Bildmuster
>* ReLU filtert relevante Merkmale weiter

>* Max-Pooling verdichtet wichtige Merkmale effizient
>* Kleine Verschiebungen stören weniger

>* Schichten lernen immer abstraktere Bildmerkmale
>* ReLU und Pooling verdichten wichtige Hinweise



In [ ]:
#@title Python-Code - Aktivierung und Pooling

# Dieses Beispiel zeigt Aktivierung und Pooling.
# ReLU entfernt negative Werte aus Merkmalskarten.
# Max-Pooling verdichtet starke lokale Bildhinweise.

import numpy as np
import torch
import matplotlib.pyplot as plt

# Ein kleines synthetisches Bild bleibt vollständig im Speicher.
image = np.zeros((8, 8), dtype=np.float32)
image[2:6, 3:5] = 1.0

# Ein einfacher Kantenfilter erzeugt positive und negative Antworten.
edge_kernel = torch.tensor(
    [[[-1.0, 0.0, 1.0], [-1.0, 0.0, 1.0], [-1.0, 0.0, 1.0]]]
)

# PyTorch erwartet die Form Batch, Kanal, Höhe, Breite.
input_tensor = torch.tensor(image).unsqueeze(0).unsqueeze(0)
conv_layer = torch.nn.Conv2d(1, 1, kernel_size=3, bias=False)

# Die Filtergewichte werden fest gesetzt, nicht trainiert.
with torch.no_grad():
    conv_layer.weight.copy_(edge_kernel.unsqueeze(0))

# Faltung, ReLU und Max-Pooling bilden eine typische CNN-Einheit.
feature_map = conv_layer(input_tensor)
activated_map = torch.relu(feature_map)
pooled_map = torch.nn.functional.max_pool2d(activated_map, kernel_size=2)

# Kleine Prüfungen machen die erwarteten Formen sichtbar.
if pooled_map.shape != (1, 1, 3, 3):
    raise ValueError("Die Pooling-Form passt nicht zum Beispiel.")

# Die Werte werden für eine einfache Darstellung vorbereitet.
activated_image = activated_map.squeeze().detach().numpy()
pooled_image = pooled_map.squeeze().detach().numpy()

print("Eingabebild: 8x8 Pixel, synthetischer heller Balken.")
print(f"Nach Faltung und ReLU: {activated_image.shape[0]}x{activated_image.shape[1]} Werte.")
print(f"Nach Max-Pooling: {pooled_image.shape[0]}x{pooled_image.shape[1]} Werte.")
print(f"Stärkste Aktivierung vor Pooling: {activated_image.max():.1f}.")
print(f"Stärkste Aktivierung nach Pooling: {pooled_image.max():.1f}.")

# Die Grafik zeigt die kompaktere Merkmalskarte nach Pooling.
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(pooled_image, cmap="viridis", vmin=0.0)

ax.set_title("Max-Pooling nach ReLU")
ax.set_xlabel("gepoolte Spalte")
ax.set_ylabel("gepoolte Zeile")
fig.colorbar(im, ax=ax, label="Aktivierungsstärke")
plt.show()



### **2.2. Flatten**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_18/Lecture_A/image_02_02.jpg?v=1787663005" width="250">



>* CNNs erzeugen gelernte Merkmalskarten.
>* Flatten macht daraus einen Dense-Vektor.

>* Flatten ordnet Merkmalskarten zu einem Vektor
>* Dense-Schichten klassifizieren erkannte Musterkombinationen

>* Vektorgröße bestimmt Dense-Eingaben korrekt
>* Flatten verbindet Merkmalskarten mit Klassifikation



In [ ]:
#@title Python-Code - Flatten

# Dieses Beispiel zeigt Flatten im LeNet-Übergang.
# Mehrdimensionale Merkmalskarten werden zu Vektoren.
# Die Ausgabeform passt danach zur Dense-Schicht.

import torch
import matplotlib.pyplot as plt

# Ein kleines Batch simuliert zwei CNN-Merkmalsausgaben.
feature_maps = torch.arange(2 * 3 * 4 * 4, dtype=torch.float32)
feature_maps = feature_maps.reshape(2, 3, 4, 4)

# Flatten behält die Batch-Achse und glättet alles danach.
flatten = torch.nn.Flatten(start_dim=1)
flat_vectors = flatten(feature_maps)

# Eine Dense-Schicht erwartet genau diese Vektorlänge.
dense_layer = torch.nn.Linear(in_features=48, out_features=10)
dense_output = dense_layer(flat_vectors)

# Diese Prüfungen machen Formfehler früh sichtbar.
assert flat_vectors.shape == (2, 48)
assert dense_output.shape == (2, 10)

print(f"Form vor Flatten: {tuple(feature_maps.shape)}")
print(f"Form nach Flatten: {tuple(flat_vectors.shape)}")
print(f"Dense-Ausgabeform: {tuple(dense_output.shape)}")
print(f"Erste 12 Werte des ersten Vektors: {flat_vectors[0, :12].tolist()}")

# Die Grafik zeigt die Reihenfolge der geglätteten Werte.
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(flat_vectors[0].numpy(), marker="o", linewidth=1)
ax.set_title("Flatten: Merkmalskarten werden ein Vektor")
ax.set_xlabel("Position im flachen Vektor")
ax.set_ylabel("Wert")
plt.show()



### **2.3. LeNet Klasse**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_18/Lecture_A/image_02_03.jpg?v=1787663003" width="250">



>* LeNet verbindet CNN-Schichten zu einem Modell
>* Faltung, Aktivierung und Pooling erzeugen Vorhersagen

>* Schichten definieren, Vorwärtsfluss separat festlegen
>* Merkmale extrahieren, abflachen und klassifizieren

>* Tensorformen und Kanäle sorgfältig abstimmen
>* LeNet als überschaubaren CNN-Einstieg nutzen



In [ ]:
#@title Python-Code - LeNet Klasse

# Dieses Beispiel baut eine kleine LeNet Klasse.
# Es zeigt Faltung, Pooling und Dense Schichten.
# Am Ende sehen wir die Tensorformen.

import torch
import matplotlib.pyplot as plt

# Eine feste Startzahl macht das Beispiel reproduzierbar.
torch.manual_seed(42)

# Die Klasse beschreibt die Schichten und den Datenfluss.
class LeNet(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = torch.nn.Conv2d(1, 6, kernel_size=5)

        self.pool = torch.nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = torch.nn.Conv2d(6, 16, kernel_size=5)
        self.fc1 = torch.nn.Linear(16 * 4 * 4, 120)

        self.fc2 = torch.nn.Linear(120, 84)
        self.fc3 = torch.nn.Linear(84, 10)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = self.pool(x)
        x = torch.relu(self.conv2(x))

        x = self.pool(x)
        x = torch.flatten(x, start_dim=1)
        x = torch.relu(self.fc1(x))

        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# Ein synthetisches Graustufenbild ersetzt einen Datensatz.
image = torch.zeros(1, 1, 28, 28)
image[:, :, 8:20, 10:18] = 1.0

# Das Modell verarbeitet ein Bild mit einem Kanal.
model = LeNet()
logits = model(image)

# Diese Prüfung schützt vor falschen Ausgabedimensionen.
if logits.shape != torch.Size([1, 10]):
    raise ValueError("Die Ausgabe sollte die Form [1, 10] haben.")

# Wir betrachten wichtige Formen entlang der Architektur.
with torch.no_grad():
    after_conv1 = torch.relu(model.conv1(image))
    after_pool1 = model.pool(after_conv1)

    after_conv2 = torch.relu(model.conv2(after_pool1))
    after_pool2 = model.pool(after_conv2)
    flattened = torch.flatten(after_pool2, start_dim=1)

print("Eingabeform:", tuple(image.shape))
print("Nach Conv1:", tuple(after_conv1.shape))
print("Nach Pool1:", tuple(after_pool1.shape))
print("Nach Conv2:", tuple(after_conv2.shape))
print("Nach Pool2:", tuple(after_pool2.shape))
print("Nach Flatten:", tuple(flattened.shape))
print("Ausgabeform:", tuple(logits.shape))

# Die Grafik zeigt das einfache synthetische Eingabebild.
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(image[0, 0].numpy(), cmap="gray", vmin=0, vmax=1)

ax.set_title("Synthetisches Eingabebild für LeNet")
ax.set_xlabel("Pixelspalte")
ax.set_ylabel("Pixelzeile")
plt.show()



## **3. Training und Fehleranalyse**

### **3.1. Training auf CPU**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_18/Lecture_A/image_03_01.jpg?v=1787662995" width="250">



>* CPU-Training macht Lernschritte gut nachvollziehbar
>* Daten, Verlust und Optimierung wirken zusammen

>* Training und Validierung klar trennen
>* CPU-Training bewusst planen und beobachten

>* Konfusionsmatrix zeigt systematische Klassenverwechslungen
>* Fehlerbilder helfen, Modellschwächen gezielt zu verbessern



In [ ]:
#@title Python-Code - Training auf CPU

# Wir trainieren ein kleines CNN bewusst auf CPU.
# Validierung zeigt Lernen ohne direkte Gewichtsänderung.
# Fehlerbilder machen falsche Vorhersagen anschaulich.

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
import sklearn

# Feste Startwerte machen das Ergebnis gut vergleichbar.
np.random.seed(42)
torch.manual_seed(42)

# Der Digits-Datensatz enthält kleine Graustufenbilder.
digits = load_digits()
images = digits.images.astype(np.float32) / 16.0
labels = digits.target.astype(np.int64)

# Eine einfache Prüfung schützt vor falschen Annahmen.
if images.shape[1:] != (8, 8):
    raise ValueError("Erwartet werden Bilder mit 8 mal 8 Pixeln.")

# PyTorch erwartet Bilddaten als Batch, Kanal, Höhe, Breite.
images = images[:, None, :, :]
train_images, valid_images, train_labels, valid_labels = train_test_split(
    images, labels, test_size=0.25, random_state=42, stratify=labels
)

# TensorDataset verbindet Bilder und Labels für Mini-Batches.
train_dataset = TensorDataset(
    torch.tensor(train_images), torch.tensor(train_labels)
)
valid_dataset = TensorDataset(
    torch.tensor(valid_images), torch.tensor(valid_labels)
)

# Kleine Batches halten das CPU-Training übersichtlich.
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=128, shuffle=False)

# Dieses LeNet-ähnliche Modell bleibt bewusst klein.
class SmallLeNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 6, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(6 * 4 * 4, 32), nn.ReLU(), nn.Linear(32, 10)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

# Das Modell und alle Tensoren bleiben auf der CPU.
device = torch.device("cpu")
model = SmallLeNet().to(device)
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# Diese Listen speichern den Verlauf pro Epoche.
train_losses = []
valid_accuracies = []

# Training verändert Gewichte, Validierung bewertet nur.
for epoch in range(5):
    model.train()
    total_loss = 0.0
    for batch_images, batch_labels in train_loader:
        optimizer.zero_grad()
        predictions = model(batch_images.to(device))
        loss = loss_function(predictions, batch_labels.to(device))
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch_images.size(0)

    model.eval()
    correct = 0
    with torch.no_grad():
        for batch_images, batch_labels in valid_loader:
            predictions = model(batch_images.to(device))
            predicted_labels = predictions.argmax(dim=1).cpu()
            correct += (predicted_labels == batch_labels).sum().item()

    train_losses.append(total_loss / len(train_dataset))
    valid_accuracies.append(correct / len(valid_dataset))

# Nach dem Training sammeln wir alle Validierungsvorhersagen.
model.eval()
with torch.no_grad():
    valid_tensor = torch.tensor(valid_images).to(device)
    all_predictions = model(valid_tensor).argmax(dim=1).cpu().numpy()

# Die Konfusionsmatrix zeigt typische Verwechslungen.
confusion = confusion_matrix(valid_labels, all_predictions)
wrong_indices = np.where(all_predictions != valid_labels)[0]

# Gedruckte Kennzahlen bleiben kurz und lernorientiert.
print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Letzter Trainingsverlust: {train_losses[-1]:.3f}")
print(f"Validierungsgenauigkeit: {valid_accuracies[-1]:.3f}")
print(f"Anzahl Fehlerbilder: {len(wrong_indices)} von {len(valid_labels)}")

# Wir zeigen ein konkretes Fehlerbild oder ein korrektes Beispiel.
if len(wrong_indices) > 0:
    shown_index = wrong_indices[0]
    title = (
        f"Fehlerbild: wahr {valid_labels[shown_index]}, "
        f"vorhergesagt {all_predictions[shown_index]}"
    )
else:
    shown_index = 0
    title = "Kein Fehler gefunden: erstes Validierungsbild"

# Eine einzelne Abbildung fokussiert die Fehleranalyse.
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(valid_images[shown_index, 0], cmap="gray", vmin=0, vmax=1)
ax.set_title(title)
ax.set_xlabel("Pixelspalte")
ax.set_ylabel("Pixelzeile")
plt.show()



### **3.2. Fehleranalyse**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_18/Lecture_A/image_03_02.jpg?v=1787662999" width="250">



>* Genauigkeit allein erklärt CNN-Fehler nicht
>* Fehler zeigen Daten- und Modellgrenzen

>* Konfusionsmatrix zeigt typische Klassenverwechslungen
>* Fehler nach Anwendungskontext unterschiedlich bewerten

>* Fehlerbilder zeigen Daten- und Modellprobleme
>* Analyse leitet gezielte Verbesserungen ein



In [ ]:
#@title Python-Code - Fehleranalyse

# Wir analysieren Fehler eines kleinen Bildklassifikators.
# Die Konfusionsmatrix zeigt typische Verwechslungen.
# Fehlerbilder machen Modellgrenzen konkret sichtbar.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import sklearn

# Wir laden kleine Ziffernbilder aus scikit-learn.
digits = load_digits()
images = digits.images
targets = digits.target

# Diese Prüfung schützt vor unerwarteten Datenformen.
if images.ndim != 3 or images.shape[1:] != (8, 8):
    raise ValueError("Erwartet werden kleine 8-mal-8-Ziffernbilder.")

# Für das Modell werden Bilder zu Merkmalsvektoren.
features = images.reshape(images.shape[0], -1)
class_names = [str(label) for label in digits.target_names]

# Die Aufteilung bleibt durch Stratifikation klassenfair.
X_train, X_valid, y_train, y_valid = train_test_split(
    features, targets, test_size=0.25, stratify=targets, random_state=42
)

# Ein einfaches Modell genügt für die Fehleranalyse.
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=300, random_state=42, solver="lbfgs")
)

# Das Modell lernt nur aus den Trainingsdaten.
model.fit(X_train, y_train)
y_pred = model.predict(X_valid)

# Wir suchen ein konkretes falsch klassifiziertes Bild.
wrong_positions = np.flatnonzero(y_pred != y_valid)
if wrong_positions.size == 0:
    raise ValueError("Dieses Modell machte hier keinen Validierungsfehler.")

# Das erste Fehlerbild wird später im Titel beschrieben.
first_wrong = int(wrong_positions[0])
true_label = int(y_valid[first_wrong])
pred_label = int(y_pred[first_wrong])

# Kurze Kennzahlen ordnen die Matrix ein.
accuracy = accuracy_score(y_valid, y_pred)
print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Validierungsgenauigkeit: {accuracy:.3f}")
print(f"Erstes Fehlerbild: wahr={true_label}, vorhergesagt={pred_label}")

# Eine Konfusionsmatrix zeigt systematische Klassenverwechslungen.
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_valid, y_pred, display_labels=class_names, cmap="Blues", ax=ax
)

ax.set_title("Konfusionsmatrix der Validierungsdaten")
ax.set_xlabel("Vorhergesagte Klasse")
ax.set_ylabel("Wahre Klasse")
plt.tight_layout()
plt.show()



### **3.3. Frameworks im Vergleich**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_18/Lecture_A/image_03_03.jpg?v=1787662997" width="250">



>* Frameworks nach gesamtem Arbeitsablauf vergleichen
>* PyTorch macht Trainingsschritte besonders transparent

>* PyTorch erleichtert flexible Fehleranalyse im Training
>* Konfusionsmatrizen zeigen häufige Klassenverwechslungen

>* Frameworkwahl richtet sich nach Anwendungskontext
>* Bewertung bleibt ein fortlaufender Verbesserungsprozess



In [ ]:
#@title Python-Code - Frameworks im Vergleich

# Wir vergleichen Auswertungsschritte in zwei Framework-Stilen.
# Eine kleine Ziffernaufgabe zeigt typische Fehleranalyse.
# Die Grafik macht Verwechslungen im Modell sichtbar.

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix

# Der Datensatz enthält kleine Graustufenbilder von Ziffern.
digits = load_digits()
images = digits.images.astype(np.float32) / 16.0
labels = digits.target.astype(np.int64)

# Eine einfache Prüfung verhindert stille Formfehler.
if images.shape[1:] != (8, 8):
    raise ValueError("Erwartet werden 8 mal 8 Pixel pro Bild.")

# Wir nutzen eine kleine, schnelle Teilmenge.
images = images[:600]
labels = labels[:600]

# Die Aufteilung bleibt durch Stratifikation fairer.
train_images, valid_images, train_labels, valid_labels = train_test_split(
    images, labels, test_size=0.25, random_state=42, stratify=labels
)

# PyTorch erwartet Bilddaten mit Kanalachse.
train_x = torch.tensor(train_images[:, None, :, :], dtype=torch.float32)
train_y = torch.tensor(train_labels, dtype=torch.long)
valid_x = torch.tensor(valid_images[:, None, :, :], dtype=torch.float32)
valid_y = torch.tensor(valid_labels, dtype=torch.long)

# Dieses kleine CNN erinnert an LeNet.
torch.manual_seed(42)
model = nn.Sequential(
    nn.Conv2d(1, 6, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(), nn.Linear(6 * 4 * 4, 10)
)

# Optimierer und Verlustfunktion steuern das Training.
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_function = nn.CrossEntropyLoss()

# Die Trainingsschleife zeigt die expliziten PyTorch-Schritte.
model.train()
for epoch in range(20):
    optimizer.zero_grad()
    logits = model(train_x)
    loss = loss_function(logits, train_y)
    loss.backward()
    optimizer.step()

# Validierung läuft ohne Gradientenberechnung.
model.eval()
with torch.no_grad():
    valid_logits = model(valid_x)
    predicted_labels = valid_logits.argmax(dim=1).numpy()

# Die Kennzahl fasst die Leistung knapp zusammen.
accuracy = accuracy_score(valid_labels, predicted_labels)
print(f"scikit-learn-Version: 1.9.0")
print(f"Validierungsgenauigkeit: {accuracy:.3f}")

# Fehlerbilder verbinden Kennzahl und konkrete Beispiele.
wrong_indices = np.where(predicted_labels != valid_labels)[0]
print(f"Falsch klassifizierte Validierungsbilder: {len(wrong_indices)}")

# Die Konfusionsmatrix zeigt systematische Verwechslungen.
confusion = confusion_matrix(valid_labels, predicted_labels, labels=np.arange(10))
fig, ax = plt.subplots(figsize=(6, 5))
image = ax.imshow(confusion, cmap="Blues")

# Achsenbeschriftungen machen die Matrix lesbar.
ax.set_title("Konfusionsmatrix eines kleinen CNN")
ax.set_xlabel("Vorhergesagte Klasse")
ax.set_ylabel("Wahre Klasse")

# Klassenmarken helfen beim Vergleichen der Ziffern.
ax.set_xticks(np.arange(10))
ax.set_yticks(np.arange(10))
fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)

# Zahlen in den Feldern erleichtern die Fehleranalyse.
for row in range(10):
    for col in range(10):
        ax.text(col, row, str(confusion[row, col]), ha="center", va="center")

plt.show()



# <font color="#418FDE" size="6.5" uppercase>**LeNet mit PyTorch**</font>


In this lecture, you learned to:
- Laden kleine Bilddatensätze mit torchvision und definieren Transformationen. 
- Implementieren ein LeNet-ähnliches CNN mit Conv2d, ReLU, Pooling und Dense-Schichten. 
- Trainieren und bewerten das CNN mit Validierung, Konfusionsmatrix und Fehlerbildern. 

In the next Lecture (Lecture B), we will go over 'Transfer mit PyTorch'